In [2]:
import pandas as pd
from pathlib import Path

# Project paths
PROJECT_ROOT = Path("..")
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "DataCoSupplyChainDataset.csv"

# Load raw dataset
df = pd.read_csv(
    DATA_PATH,
    encoding="ISO-8859-1"
)

print("Shape:", df.shape)
df.head()

Shape: (180519, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [3]:
date_columns = [
    "order date (DateOrders)",
    "shipping date (DateOrders)",
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df[date_columns].dtypes

order date (DateOrders)       datetime64[us]
shipping date (DateOrders)    datetime64[us]
dtype: object

In [4]:
quality_overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_count": df.nunique(dropna=False),
})

quality_overview.sort_values(
    ["missing_pct", "unique_count"],
    ascending=[False, True]
)

,dtype,missing_count,missing_pct,unique_count
Product Description,float64,180519,100.00,1
Order Zipcode,float64,155679,86.24,610
Customer Email,str,0,0.00,1
Customer Password,str,0,0.00,1
Product Status,int64,0,0.00,1
Late_delivery_risk,int64,0,0.00,2
Customer Country,str,0,0.00,2
Customer Segment,str,0,0.00,3
Type,str,0,0.00,4
Days for shipment (scheduled),int64,0,0.00,4


In [5]:
category_mapping = (
    df[["Category Id", "Category Name"]]
    .drop_duplicates()
    .sort_values(["Category Name", "Category Id"])
)

duplicate_category_names = (
    category_mapping
    .groupby("Category Name")
    .filter(lambda x: len(x) > 1)
)

duplicate_category_names

,Category Id,Category Name
55,13,Electronics
62,37,Electronics


In [6]:
duplicate_rows = df.duplicated().sum()
print("Exact duplicate rows:", duplicate_rows)

order_item_counts = (
    df.groupby("Order Id")
      .size()
      .describe()
)

print("\nItems per order:")
print(order_item_counts)

multi_item_orders = (
    df.groupby("Order Id")
      .size()
      .gt(1)
      .sum()
)

print("\nOrders with more than one row/item:", multi_item_orders)

target_per_order = (
    df.groupby("Order Id")["Late_delivery_risk"]
      .nunique()
)

print(
    "\nOrders with inconsistent target values:",
    (target_per_order > 1).sum()
)

Exact duplicate rows: 0

Items per order:
count    65752.000000
mean         2.745453
std          1.478363
min          1.000000
25%          1.000000
50%          3.000000
75%          4.000000
max          5.000000
dtype: float64

Orders with more than one row/item: 45902

Orders with inconsistent target values: 0


In [7]:
order_customer_check = (
    df.groupby("Order Id")["Customer Id"]
      .nunique()
)


In [8]:
print(
    "Orders linked to more than one customer:",
    (order_customer_check > 1).sum()
)

Orders linked to more than one customer: 0


In [9]:
df.loc[
    df["Category Id"].isin([13, 37]),
    [
        "Category Id",
        "Category Name",
        "Department Id",
        "Department Name",
        "Product Card Id",
        "Product Name"
    ]
].drop_duplicates().sort_values(
    ["Category Id", "Department Id", "Product Card Id"]
)

,Category Id,Category Name,Department Id,Department Name,Product Card Id,Product Name
303,13,Electronics,3,Footwear,273,Under Armour Kids' Mercenary Slide
482,13,Electronics,3,Footwear,276,Under Armour Women's Ignite Slide
55,13,Electronics,3,Footwear,278,Under Armour Men's Compression EV SL Slide
308,13,Electronics,3,Footwear,282,Under Armour Women's Ignite PIP VI Slide
115,37,Electronics,6,Outdoors,818,Titleist Pro V1x Golf Balls
755,37,Electronics,6,Outdoors,821,Titleist Pro V1 High Numbers Personalized Gol
361,37,Electronics,6,Outdoors,822,Titleist Pro V1x High Numbers Golf Balls
505,37,Electronics,6,Outdoors,823,Titleist Pro V1x High Numbers Personalized Go
116,37,Electronics,6,Outdoors,825,Bridgestone e6 Straight Distance NFL Tennesse
62,37,Electronics,6,Outdoors,828,Bridgestone e6 Straight Distance NFL San Dieg


In [10]:
keep_and_key_features = [
    "Type",
    "Category Id",
    "Customer Segment",
    "Customer State",
    "Department Name",
    "Order Country",
    "Order Item Discount",
    "Order Item Quantity",
    "Order Region",
    "Product Name",
    "Shipping Mode",
    "Order Id",
    "Customer Id",
    "order date (DateOrders)",
]

for col in keep_and_key_features:
    print(f"\n--- {col} ---")
    print("dtype:", df[col].dtype)
    print("missing:", df[col].isna().sum())
    print("unique:", df[col].nunique(dropna=False))

    if df[col].dtype == "object" or str(df[col].dtype).startswith("string"):
        print(df[col].value_counts(dropna=False).head(20))
    else:
        print(df[col].describe())


--- Type ---
dtype: str
missing: 0
unique: 4
count     180519
unique         4
top        DEBIT
freq       69295
Name: Type, dtype: object

--- Category Id ---
dtype: int64
missing: 0
unique: 51
count    180519.000000
mean         31.851451
std          15.640064
min           2.000000
25%          18.000000
50%          29.000000
75%          45.000000
max          76.000000
Name: Category Id, dtype: float64

--- Customer Segment ---
dtype: str
missing: 0
unique: 3
count       180519
unique           3
top       Consumer
freq         93504
Name: Customer Segment, dtype: object

--- Customer State ---
dtype: str
missing: 0
unique: 46
count     180519
unique        46
top           PR
freq       69373
Name: Customer State, dtype: object

--- Department Name ---
dtype: str
missing: 0
unique: 11
count       180519
unique          11
top       Fan Shop
freq         66861
Name: Department Name, dtype: object

--- Order Country ---
dtype: str
missing: 0
unique: 164
count             180519


## Raw Feature Action List

### KEEP / RETAIN

1. `Type`: **KEEP** — categorical; use in baseline.

2. `Category Id`: **KEEP** — categorical code; do not treat as continuous numeric.

3. `Customer Segment`: **KEEP** — low-cardinality categorical feature.

4. `Customer State`: **KEEP + CLEAN** — main customer geography feature; convert 3 invalid ZIP-like values to missing.

5. `Department Name`: **KEEP** — human-readable department representation.

6. `Order Country`: **KEEP** — destination-country categorical feature.

7. `Order Item Discount`: **KEEP** — primary discount representation.

8. `Order Item Quantity`: **KEEP** — low-cardinality order-time feature.

9. `Order Region`: **KEEP** — lower-cardinality geographic representation.

10. `Product Name`: **KEEP** — categorical product representation.

11. `Shipping Mode`: **KEEP** — main shipping-service representation; strong target association.

12. `Customer Id`: **RETAIN TECHNICALLY / DROP FROM MODEL** — use only for grouping and split checks.

13. `Order Id`: **RETAIN TECHNICALLY / DROP FROM MODEL** — required for order-level grouping and leakage-safe splitting.

14. `order date (DateOrders)`: **RETAIN FOR FEATURE ENGINEERING / DROP AS RAW MODEL FEATURE** — source for time features such as hour, day-of-week, and month.

15. `Late_delivery_risk`: **TARGET** — prediction label only; never use as model input.

In [11]:
df["Type"].value_counts(dropna=False)

Type
DEBIT       69295
TRANSFER    49883
PAYMENT     41725
CASH        19616
Name: count, dtype: int64

In [12]:
pd.crosstab(
    df["Type"],
    df["Late_delivery_risk"],
    normalize="index"
).round(3)

Late_delivery_risk,0,1
Type,,
CASH,0.434,0.566
DEBIT,0.428,0.572
PAYMENT,0.425,0.575
TRANSFER,0.515,0.485


In [13]:
df["Order Country"].value_counts(dropna=False).describe()

count      164.000000
mean      1100.725610
std       2816.940664
min          1.000000
25%         30.500000
50%        157.500000
75%        759.500000
max      24840.000000
Name: count, dtype: float64

In [17]:
df.shape

(180519, 53)

In [14]:
df["Order Country"].value_counts(dropna=False).sort_values().head(15)

Order Country
Serbia                 1
Burundi                1
Kuwait                 2
Eritrea                2
Sáhara Occidental      2
Guinea Ecuatorial      2
Chad                   3
Baréin                 4
Bután                  5
Armenia                5
República de Gambia    5
Suazilandia            5
Sudán del Sur          5
Eslovenia              6
Macedonia              6
Name: count, dtype: int64

In [18]:
df["Order Item Discount"].describe()

count    180519.000000
mean         20.664741
std          21.800901
min           0.000000
25%           5.400000
50%          14.000000
75%          29.990000
max         500.000000
Name: Order Item Discount, dtype: float64

In [19]:
(df["Order Item Discount"] > df["Sales"]).value_counts()

False    180519
Name: count, dtype: int64

In [20]:
threshold = 29.99

high_discount = df[df["Order Item Discount"] > threshold]

print("Rows above 29.99:", len(high_discount))
print(
    "Percentage:",
    round(len(high_discount) / len(df) * 100, 2),
    "%"
)

Rows above 29.99: 45112
Percentage: 24.99 %


In [21]:
high_discount["Order Item Discount"].describe()

count    45112.000000
mean        50.133130
std         23.252669
min         30.000000
25%         36.000000
50%         45.000000
75%         59.990002
max        500.000000
Name: Order Item Discount, dtype: float64

In [22]:
high_discount["Order Item Discount"].sort_values(ascending=False).head(30)

12606     500.0
59847     400.0
126395    375.0
40387     375.0
169211    375.0
42486     375.0
123298    375.0
68538     375.0
45115     375.0
45116     375.0
148683    375.0
160132    375.0
93383     375.0
159170    375.0
120294    375.0
120292    375.0
82082     375.0
82079     375.0
50314     375.0
126394    375.0
125500    375.0
12559     375.0
17982     375.0
134242    375.0
1594      375.0
12558     375.0
170036    375.0
82161     360.0
141117    340.0
159182    320.0
Name: Order Item Discount, dtype: float64

In [26]:
df.loc[
    df["Order Item Discount"] >= 300,
    [
        "Order Item Discount",
        "Sales",
        "Order Item Discount Rate",
        "Order Item Product Price",
        "Order Item Quantity",
        "Product Name"
    ]
].sort_values(
    "Order Item Discount",
    ascending=False
).head(30)

,Order Item Discount,Sales,Order Item Discount Rate,Order Item Product Price,Order Item Quantity,Product Name
12606,500.0,1999.98999,0.25,1999.98999,1,SOLE E35 Elliptical
59847,400.0,1999.98999,0.20,1999.98999,1,SOLE E35 Elliptical
125500,375.0,1500.00000,0.25,1500.00000,1,Dell Laptop
45115,375.0,1500.00000,0.25,1500.00000,1,Dell Laptop
126395,375.0,1500.00000,0.25,1500.00000,1,Dell Laptop
1594,375.0,1500.00000,0.25,1500.00000,1,Dell Laptop
82082,375.0,1500.00000,0.25,1500.00000,1,Dell Laptop
82079,375.0,1500.00000,0.25,1500.00000,1,Dell Laptop
134242,375.0,1500.00000,0.25,1500.00000,1,Dell Laptop
68538,375.0,1500.00000,0.25,1500.00000,1,Dell Laptop
